# Lekcja 1. Stan kubitu

**Cel:** zrozumieć, czym jest kubit fizycznie, czym jest jego stan matematycznie i dlaczego to dwie różne rzeczy.

**Notacja:** piszemy w języku algebry liniowej. Wektory bazy standardowej w $\mathbb{C}^2$ to $e_0 = (1, 0)$ i $e_1 = (0, 1)$. Fizycy oznaczają je $|0\rangle$ i $|1\rangle$; tej notacji używamy tylko jako etykiet, bo tak opisują wyniki biblioteki (PennyLane, Qiskit).

## 1. Kubit to układ fizyczny

**Układ dwupoziomowy** to układ kwantowy, w którym liczą się dokładnie dwa stany o różnej energii:

- **stan podstawowy** (najniższa energia), który umownie nazywamy $e_0$;
- **stan wzbudzony**, który nazywamy $e_1$.

Nazwy „0” i „1” to umowa. Fizycznie to po prostu „niższa” i „wyższa” energia.

Niektóre układy mają dokładnie dwa stany (spin elektronu, polaryzacja fotonu). Większość ma ich więcej, a my **wybieramy dwa najniższe** i pilnujemy, żeby układ nie wszedł na trzeci.

### Transmon, czyli kubit nadprzewodzący (IBM Heron, Google Willow)

To obwód elektryczny na chipie: kondensator i **złącze Josephsona** (dwa nadprzewodniki rozdzielone cienką warstwą izolatora). Schłodzony zachowuje się jak sztuczny atom: ma dyskretne poziomy energii.

Kluczowe jest to, że poziomy są rozłożone **nierówno** (anharmoniczność). Przejście $0 \to 1$ ma inną częstotliwość niż $1 \to 2$, więc impuls mikrofalowy dostrojony do $0 \leftrightarrow 1$ prawie nie wzbudza poziomu 2. Anharmoniczność to jednak tylko ok. −300 MHz, czyli kilka procent częstotliwości kubitu, dlatego impulsy się kształtuje (np. DRAG), żeby ten wyciek tłumić. Gdyby poziomy były równo rozłożone (zwykły oscylator harmoniczny), każdy impuls przenoszący $0 \to 1$ przenosiłby też $1 \to 2$, $2 \to 3$ itd. i nie dałoby się wyizolować dwóch poziomów.

Kontrola nie jest doskonała: czasem stan ucieka na poziom 2 (*leakage*). To jedno ze źródeł błędów.

### Po co chłodzenie do ok. 15 mK?

Z trzech powodów:

1. **Nadprzewodnictwo.** Aluminium w złączach przechodzi w stan nadprzewodzący dopiero poniżej ok. 1,2 K. Bez tego nie ma kubitu.
2. **Znany punkt startowy.** Energia przejścia $0 \to 1$ (ok. 5 GHz, czyli ok. 0,24 K) musi być dużo większa od energii drgań termicznych. W 15 mK otoczenie jest kilkanaście razy „za słabe”, żeby przypadkowo wzbudzić kubit, więc kubit sam opada do $e_0$ i tam zostaje.
3. **Cisza w trakcie obliczeń.** Każde przypadkowe wzbudzenie to błąd. Niska temperatura wydłuża czas, przez który kubit trzyma swój stan.

### Jak przygotować stan początkowy $e_0 \otimes \dots \otimes e_0$?

- **Czekanie:** wzbudzony kubit sam opada do $e_0$ (relaksacja, czas $T_1$ rzędu 100 µs). Wystarczy odczekać kilka $T_1$.
- **Aktywny reset (szybciej):** zmierz kubit; jeśli wyszło 1, zastosuj bramkę X.

W praktyce kubit jest w $e_0$ z prawdopodobieństwem ok. 99%, nie 100%. To część tzw. błędów SPAM (*state preparation and measurement*). Każdy inny stan początkowy buduje się z $e_0$ bramkami.

## 2. Stan kubitu to wektor jednostkowy w $\mathbb{C}^2$

$$v = \alpha e_0 + \beta e_1 = (\alpha, \beta), \qquad \alpha, \beta \in \mathbb{C}, \qquad |\alpha|^2 + |\beta|^2 = 1.$$

- $\alpha$ i $\beta$ to **amplitudy**: współrzędne wektora stanu w bazie $\{e_0, e_1\}$. Są zespolone i nie są prawdopodobieństwami.
- Warunek normalizacji mówi tylko, że $\|v\| = 1$.
- Uwaga na moduł: $|i|^2 = 1$, a nie $i^2 = -1$.
- Stan jest wyznaczony z dokładnością do **fazy globalnej**: $v$ i $e^{i\gamma} v$ opisują ten sam stan fizyczny (szczegóły w lekcji 3). Formalnie stan to promień, a nie wektor.

Para $(\alpha, \beta)$ to cztery liczby rzeczywiste niezależnie od zapisu (kartezjańskiego $a + bi$ czy biegunowego $r e^{i\varphi}$). Zapis zmienia, które cztery liczby widać, ale nie ile ich jest.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
GREEN, DARK, LIGHT, GREY, ORANGE = "#44B749", "#1A8A1F", "#96E199", "#8A8A8A", "#E08A00"

e0 = np.array([1, 0], dtype=complex)   # etykieta |0⟩
e1 = np.array([0, 1], dtype=complex)   # etykieta |1⟩

def norm(v):
    return np.sqrt(np.sum(np.abs(v) ** 2))

def show(v, n=None):
    """Wypisz wektor stanu z etykietami bitów (etykieta = pozycja w zapisie dwójkowym)."""
    v = np.asarray(v)
    n = n or int(np.log2(len(v)))
    for k, a in enumerate(v):
        if abs(a) > 1e-12:
            print(f"  pozycja {k} ({format(k, f'0{n}b')}): amplituda {np.round(a, 3)}, P = {abs(a)**2:.3f}")

# Które wektory mogą być stanem kubitu? Wystarczy norma 1.
kandydaci = {
    "(1/2, 1/2)":       np.array([1/2, 1/2]),
    "(1/√2, i/√2)":     np.array([1, 1j]) / np.sqrt(2),
    "(3/5, 4/5)":       np.array([3/5, 4/5]),
    "(1, 1)":           np.array([1, 1]),
}
for nazwa, v in kandydaci.items():
    ok = np.isclose(norm(v), 1)
    print(f"{nazwa:>14}: ||v||² = {norm(v)**2:.3f}  ->  {'stan' if ok else 'NIE jest stanem'}")

    (1/2, 1/2): ||v||² = 0.500  ->  NIE jest stanem
  (1/√2, i/√2): ||v||² = 1.000  ->  stan
    (3/5, 4/5): ||v||² = 1.000  ->  stan
        (1, 1): ||v||² = 2.000  ->  NIE jest stanem

## 3. Trzy rzeczy, których nie wolno mylić

> **To było trudne:** słowa „kubit”, „stan” i „wynik” łatwo się zlewają. To trzy różne obiekty.

| | Kubit | Stan | Wynik pomiaru |
|---|---|---|---|
| Czym jest | układ fizyczny (transmon, jon, foton) | wektor $(\alpha, \beta) \in \mathbb{C}^2$ | liczba 0 albo 1 |
| Czy jest losowy | nie | **nie**, jest określony | **tak** |
| Czy się zmienia | nie | tak: zmieniają go bramki i pomiar | powstaje dopiero przy pomiarze |
| Czy da się go odczytać | – | nie bezpośrednio | tak, to właśnie widzimy |

**Amplitudy nie są własnością kubitu jak masa.** Z trzech powodów:

1. **Stan się zmienia:** ten sam kubit raz jest w $e_0$, po bramce w $(e_0 + e_1)/\sqrt{2}$, po pomiarze znów w $e_0$ albo $e_1$.
2. **Współrzędne zależą od bazy:** ten sam stan ma w innej bazie inne $\alpha, \beta$.
3. **Nie da się ich odczytać:** jeden pomiar daje jeden bit, a nie $\alpha$ czy $\beta$.

Dobra intuicja: kubit to „pojemnik”, a stan to jego aktualna zawartość, opisana w wybranym układzie współrzędnych.

**Przed pomiarem kubit nie jest „naprawdę” w 0 albo w 1.** Jest w stanie $(\alpha, \beta)$, który nie jest ani zerem, ani jedynką. Pomiar nie odczytuje gotowej wartości, tylko ją tworzy (lekcja 2), a doświadczenie z lekcji 3 pokazuje, że superpozycję da się odróżnić od „ukrytego 0 albo 1”. To, że żaden model z ukrytymi wartościami nie działa, pokazuje dopiero twierdzenie Bella (lekcja 7).

## 4. Dwa różne „obwody”

| | Obwód nadprzewodzący | Obwód kwantowy (*quantum circuit*) |
|---|---|---|
| Czym jest | fizyczny element na chipie | program: opis obliczenia |
| Odpowiada | **jednemu kubitowi** | **całemu algorytmowi** |
| Składa się z | kondensatora i złącza Josephsona | przewodów (kubitów) i bramek w kolejności |
| Gdzie żyje | w lodówce, 15 mK | w kodzie (`qml.qnode`, `QuantumCircuit`) |

Nazwa *circuit* w bibliotekach pochodzi od schematów logicznych. Ten sam program działa na symulatorze (gdzie nie ma żadnych fizycznych obwodów, tylko mnożenie macierzy) i na chipie. Program z 3 kubitami na IBM Heron zajmie zwykle 3 ze 156 fizycznych kubitów, które wybierze transpiler. Jeśli wybrane kubity nie są bezpośrednio połączone, transpiler doda bramki SWAP przez inne kubity. To są kubity **fizyczne**, zaszumione, a nie logiczne z korekcją błędów.

## Podsumowanie

> Kubit to fizyczny układ dwupoziomowy, schłodzony tak, żeby był nadprzewodnikiem i startował w stanie podstawowym. Jego stan opisujemy wektorem jednostkowym $v = (\alpha, \beta) \in \mathbb{C}^2$, z dokładnością do fazy globalnej. Amplitudy $\alpha, \beta$ to współrzędne tego wektora w bazie $\{e_0, e_1\}$, a nie stałe cechy kubitu.

## Zadania

1. Które z wektorów $(1/2, 1/2)$, $(1/\sqrt{2}, i/\sqrt{2})$, $(3/5, 4/5)$, $(1, 1)$ mogą być stanem kubitu? (Sprawdź wynik komórki wyżej: są **dwa** takie wektory.)
2. Stan $(\sqrt{3}/2, 1/2)$. Ile wynoszą $|\alpha|^2$ i $|\beta|^2$?
3. Dlaczego transmon, który ma wiele poziomów energii, da się używać jako kubitu? Co by się stało przy równo rozłożonych poziomach?
4. Kolega mówi: „Nasz obwód ma 3 kubity, więc na IBM Heron zajmie 3 ze 156 obwodów nadprzewodzących”. Co jest w tym poprawne, a co jest skrótem myślowym?